In [2]:
import requests
import json
import time
import math
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def create_session():
    session = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json",
        "Connection": "keep-alive"
    })
    return session

session = create_session()

def crawl_winmart(slug):
    all_items = []
    page = 1
    page_size = 20

    while True:
        url = (
            f"https://api-crownx.winmart.vn/it/api/web/v3/item/category"
            f"?orderByDesc=true&pageNumber={page}&pageSize={page_size}"
            f"&slug={slug}&storeGroupCode=1998"
        )

        try:
            res = session.get(url, timeout=(5, 30))
            data = res.json()
        except Exception as e:
            print(f"  ⚠️ Lỗi request trang {page}: {e}")
            break

        items = data.get("data", {}).get("items", [])

        if not items:
            print(f"  ⚠️ Trang {page} không có items, dừng")
            break

        all_items.extend(items)

        # ✅ paging nằm ở cấp 1: data["paging"]
        paging = data.get("paging", {})
        total_items = paging.get("totalItems") or paging.get("total") or 0
        total_pages = math.ceil(total_items / page_size) if total_items else 0

        # Fallback nếu vẫn không có
        if not total_pages:
            total_pages = page + 1 if len(items) >= page_size else page

        print(f"  [{slug}] Trang {page}/{total_pages} - Tổng: {len(all_items)}/{total_items} sản phẩm")

        if page >= total_pages:
            break

        page += 1
        time.sleep(0.3)

    with open(f"{slug}.json", "w", encoding="utf-8") as f:
        json.dump(all_items, f, ensure_ascii=False, indent=2)

    print(f"  ✅ Xong! Đã lưu {len(all_items)} sản phẩm vào {slug}.json")
    return all_items

# Đọc slug từ file
with open("slug.txt", "r", encoding="utf-8") as f:
    slugs = [line.strip() for line in f if line.strip()]

print(f"📋 Tìm thấy {len(slugs)} danh mục")

failed = []

for i, slug in enumerate(slugs, 1):
    print(f"\n[{i}/{len(slugs)}] Đang cào: {slug}")
    try:
        crawl_winmart(slug)
    except Exception as e:
        print(f"  ⚠️ Bỏ qua [{slug}]: {e}")
        failed.append(slug)
    time.sleep(0.5)

print(f"\n{'='*40}")
print(f"✅ Thành công: {len(slugs) - len(failed)}/{len(slugs)} danh mục")
if failed:
    print(f"❌ Thất bại ({len(failed)} danh mục):")
    for s in failed:
        print(f"   - {s}")
    with open("slug_failed.txt", "w", encoding="utf-8") as f:
        f.write('\n'.join(failed))
    print(f"💾 Đã lưu slug lỗi vào slug_failed.txt")

📋 Tìm thấy 92 danh mục

[1/92] Đang cào: banh-bao--c01156&storeCode=1535
  [banh-bao--c01156&storeCode=1535] Trang 1/1 - Tổng: 8/0 sản phẩm
  ✅ Xong! Đã lưu 8 sản phẩm vào banh-bao--c01156&storeCode=1535.json

[2/92] Đang cào: banh-keo--c07&storeCode=1535
  [banh-keo--c07&storeCode=1535] Trang 1/2 - Tổng: 20/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 2/3 - Tổng: 40/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 3/4 - Tổng: 60/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 4/5 - Tổng: 80/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 5/6 - Tổng: 100/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 6/7 - Tổng: 120/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 7/8 - Tổng: 140/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 8/9 - Tổng: 160/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 9/10 - Tổng: 180/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 10/11 - Tổng: 200/0 sản phẩm
  [banh-keo--c07&storeCode=1535] Trang 11/12 - Tổng: 220/0 sản phẩm
  [banh-keo--c07&

In [1]:
import json
import os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

# 📁 Đổi thành folder chứa file JSON của bạn
folder_path = "D:\\a"

all_rows = []
seen_item_nos = set()  # Tránh trùng lặp

for filename in os.listdir(folder_path):
    if not filename.endswith(".json"):
        continue

    filepath = os.path.join(folder_path, filename)
    category_name = filename.replace(".json", "")

    with open(filepath, encoding='utf-8') as f:
        data = json.load(f)

    if not data:
        continue

    for item in data:
        item_no = item.get('itemNo', '')

        if item_no and item_no in seen_item_nos:
            continue
        if item_no:
            seen_item_nos.add(item_no)

        seo = item.get('seoName', '')
        url = f"https://www.winmart.vn/products/{seo}" if seo else ''

        all_rows.append([
            category_name,
            item_no,
            item.get('name', ''),
            item.get('shortDescription', ''),
            item.get('brandName', ''),
            item.get('uomName', ''),
            item.get('barcode', ''),
            item.get('price', ''),
            item.get('salePrice', ''),
            item.get('quantity', ''),
            item.get('mch1Name', ''),
            item.get('mch2Name', ''),
            item.get('mch3Name', ''),
            item.get('mch4Name', ''),
            item.get('mch5Name', ''),
            item.get('categoryName', ''),
            url
        ])

print(f"📦 Tổng sản phẩm: {len(all_rows)}")

# 📊 Tạo Excel
wb = Workbook()
ws = wb.active
ws.title = "All Products"

headers = [
    'Danh mục (file)', 'Mã SP', 'Tên sản phẩm', 'Mô tả ngắn',
    'Thương hiệu', 'Đơn vị', 'Barcode',
    'Giá gốc', 'Giá sale',
    'Tồn kho',
    'Ngành hàng 1', 'Ngành hàng 2', 'Nhóm hàng 3', 'Nhóm hàng 4', 'Nhóm hàng 5',
    'Category', 'Link'
]
ws.append(headers)

# Style header
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(bold=True, color="FFFFFF", size=11)
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center")
ws.row_dimensions[1].height = 22

# Ghi dữ liệu
for row_data in all_rows:
    ws.append(row_data)
    last_row = ws.max_row
    url = row_data[-1]
    name_cell = ws.cell(row=last_row, column=3)
    if url and isinstance(url, str) and url.startswith("http"):
        name_cell.hyperlink = url
        name_cell.font = Font(color="0563C1", underline="single")

# Độ rộng cột
column_widths = {
    'A': 30,  # Danh mục file
    'B': 12,  # Mã SP
    'C': 40,  # Tên sản phẩm
    'D': 35,  # Mô tả ngắn
    'E': 20,  # Thương hiệu
    'F': 10,  # Đơn vị
    'G': 16,  # Barcode
    'H': 12,  # Giá gốc
    'I': 12,  # Giá sale
    'J': 10,  # Tồn kho
    'K': 20,  # Ngành hàng 1
    'L': 20,  # Ngành hàng 2
    'M': 25,  # Nhóm hàng 3
    'N': 25,  # Nhóm hàng 4
    'O': 25,  # Nhóm hàng 5
    'P': 25,  # Category
    'Q': 50,  # Link
}
for col, width in column_widths.items():
    ws.column_dimensions[col].width = width

ws.freeze_panes = "A2"

wb.save("sp.xlsx")
print(f"✅ Xong! Tổng {len(all_rows)} sản phẩm → sp.xlsx")

📦 Tổng sản phẩm: 3253
✅ Xong! Tổng 3253 sản phẩm → sp.xlsx
